# **Entrega 1 - Álgebra Linear, Vetores e Geometria Análitica**

In [ ]:
!pip install xlsxwriter

In [ ]:
import numpy as np
from PIL import Image
import pandas as pd
from google.colab import files
import io
import xlsxwriter

print("Por favor, selecione a imagem")
uploaded = files.upload()

if uploaded:
    caminho_da_imagem = next(iter(uploaded))
    image_bytes = uploaded[caminho_da_imagem]

    # Carregar a imagem
    img_pil = Image.open(io.BytesIO(image_bytes))

    #Processar dois fluxos: RGB para as cores e L (Grayscale) para os IDs numéricos
    # Redimensionamos ambos para 200x200 para manter a paridade
    img_rgb = img_pil.convert('RGB').resize((200, 200), Image.NEAREST)
    img_gray = img_pil.convert('L').resize((200, 200), Image.NEAREST)

    # 3. Converter em matrizes NumPy
    matriz_cores = np.array(img_rgb)
    matriz_numeros = np.array(img_gray)

    # 4. Iniciar a criação do arquivo Excel
    nome_saida_excel = 'matriz_colorida_liderancas.xlsx'
    workbook = xlsxwriter.Workbook(nome_saida_excel)
    worksheet = workbook.add_worksheet("Mapa de Lideranças")

    # Ajustar dimensões das células para ficarem quadradas
    # set_column(primeira_col, ultima_col, largura)
    worksheet.set_column(0, 199, 2.5)

    # Criar um cache de formatos para evitar lentidão
    # O Excel tem um limite de estilos únicos, mas como muitas células
    # compartilham a mesma cor exata, o cache otimiza o arquivo.
    formatos_cache = {}

    print("Pintando as células e ocultando os valores... Isso pode levar alguns segundos.")

    for r in range(200):
        # Ajusta a altura da linha para manter o aspecto quadrado
        worksheet.set_row(r, 15)
        for c in range(200):
            # Extrair o RGB do pixel atual e converter para Hexadecimal (#RRGGBB)
            rgb = matriz_cores[r, c]
            hex_color = '#{:02x}{:02x}{:02x}'.format(rgb[0], rgb[1], rgb[2])

            # O valor numérico que ficará "atrás" da cor
            valor_id = int(matriz_numeros[r, c])

            # Verificar se já criamos este estilo de cor para economizar memória
            if hex_color not in formatos_cache:
                formatos_cache[hex_color] = workbook.add_format({
                    'bg_color': hex_color,
                    'num_format': ';;;' # O "truque": oculta o valor na célula, mas mantém na barra de fórmulas
                })

            # Escrever o número com o formato de cor correspondente
            worksheet.write(r, c, valor_id, formatos_cache[hex_color])

    workbook.close()

    # 5. Salvar também o CSV de backup (apenas números) como você fazia antes
    df_csv = pd.DataFrame(matriz_numeros)
    df_csv.to_csv('matriz_200x200_liderancas.csv', index=False, header=False)

    print(f"\n--- SUCESSO! ---")
    print(f"1. Excel Colorido: '{nome_saida_excel}'")
    print(f"2. CSV de Números: 'matriz_200x200_liderancas.csv'")

    # Baixar automaticamente o arquivo Excel gerado
    files.download(nome_saida_excel)
else:
    print("Nenhum arquivo foi selecionado.")